# 02 — Silver Cleaning
## Bronze Delta Table → Cleaned Silver Delta Table

**Purpose:** Clean, deduplicate, enforce types, and standardise the raw Bronze data.  
No business logic or derived columns yet — that is Notebook 03.  
Every cleaning decision is documented with a reason.

**Input :** `sales_bronze.raw_superstore`  
**Output:** `sales_silver.clean_superstore`  
**Layer  :** Silver (clean)


## 1. Load config

In [0]:
%run ./00_setup_and_config

In [0]:
log("INFO", "02_silver_clean", "Starting Silver cleaning layer")


## 2. Read from Bronze Delta Table

In [0]:
df_bronze = spark.table(BRONZE_TABLE)

log("INFO", "02_silver_clean", f"Read from Bronze: {BRONZE_TABLE}")
print(f"  Rows read    : {df_bronze.count():,}")
print(f"  Columns      : {len(df_bronze.columns)}")
print(f"  Columns list : {df_bronze.columns}")


## 3. Drop Bronze metadata columns before cleaning

In [0]:
BRONZE_META_COLS = ["_source_file", "_ingested_at", "_pipeline_name"]

df = df_bronze.drop(*BRONZE_META_COLS)

print(f"  Columns after dropping Bronze metadata: {len(df.columns)}")
print(f"  Remaining columns: {df.columns}")


## 4. Remove exact duplicate rows

In [0]:
rows_before = df.count()
df = df.dropDuplicates()
rows_after  = df.count()
dupes_found = rows_before - rows_after

print(f"  Rows before deduplication : {rows_before:,}")
print(f"  Rows after deduplication  : {rows_after:,}")
print(f"  Duplicate rows removed    : {dupes_found:,}")

if dupes_found == 0:
    print("  ✅ No exact duplicates found — data is clean at this level")
else:
    log("WARN", "02_silver_clean", f"Removed {dupes_found} duplicate rows")


## 5. Re-run the type-aware null audit on Bronze data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

string_cols = {
    f.name for f in df.schema.fields
    if isinstance(f.dataType, StringType)
}

null_exprs = [
    F.count(
        F.when(
            F.col(c).isNull() | (F.col(c) == "") if c in string_cols
            else F.col(c).isNull(),
            c
        )
    ).alias(c)
    for c in df.columns
]

null_counts_before = df.select(null_exprs).collect()[0]

print("=== NULL BASELINE (entering Silver) ===\n")
problem_cols = []
for col_name, null_count in zip(df.columns, null_counts_before):
    if null_count > 0:
        print(f"  ⚠️  {col_name:<30} : {null_count} nulls/blanks")
        problem_cols.append(col_name)

if not problem_cols:
    print("  ✅ No nulls detected entering Silver")
else:
    print(f"\n  Columns needing attention : {problem_cols}")


## 6. Handle nulls in string columns

In [0]:
# Strategy per column — every decision documented:
#
#   customer_name  → fill with "Unknown Customer"
#                    Reason: downstream dim_customer needs a name
#
#   product_name   → fill with "Unknown Product"
#                    Reason: downstream dim_product needs a name
#
#   city / state   → fill with "Unknown"
#                    Reason: region analysis still works at Region level
#
#   segment        → fill with "Unknown"
#                    Reason: segment is categorical, Unknown is valid category
#
#   ship_mode      → fill with "Unknown"
#                    Reason: shipping analysis can filter Unknown separately
#
#   sub_category   → fill with "Unknown"
#                    Reason: category is still valid for higher-level analysis

from pyspark.sql import functions as F

string_fill_map = {
    "customer_name" : "Unknown Customer",
    "product_name"  : "Unknown Product",
    "city"          : "Unknown",
    "state"         : "Unknown",
    "segment"       : "Unknown",
    "ship_mode"     : "Unknown",
    "sub_category"  : "Unknown",
}

df = df.fillna(string_fill_map)

log("INFO", "02_silver_clean", f"String nulls filled with placeholder values")
print("  ✅ String null handling complete")


## 7. Handle nulls in numeric columns

In [0]:
#   sales     → drop the row
#              Reason: a transaction with no sales value is meaningless.
#              We cannot fill it with 0 (that's a real value meaning free).
#              We cannot fill it with a median (that's fabricating revenue).
#              Dropping is the only honest option.
#
#   profit    → fill with 0.0
#              Reason: some transactions genuinely break even.
#              0 is a valid and common profit value.
#
#   discount  → fill with 0.0
#              Reason: no discount recorded means no discount applied.
#              0 is the correct semantic value.
#
#   quantity  → drop the row
#              Reason: a transaction with unknown quantity cannot be
#              used for any meaningful aggregation.

rows_before_numeric = df.count()

# Drop rows where critical numeric columns are null
df = df.filter(
    F.col("sales").isNotNull() &
    F.col("quantity").isNotNull()
)

# Fill non-critical numeric nulls
df = df.fillna({
    "profit"   : 0.0,
    "discount" : 0.0,
})

rows_after_numeric = df.count()
dropped = rows_before_numeric - rows_after_numeric

print(f"  Rows dropped (null sales/quantity) : {dropped:,}")
print(f"  Rows remaining                     : {rows_after_numeric:,}")
print(f"  profit nulls filled with           : 0.0")
print(f"  discount nulls filled with         : 0.0")
log("INFO", "02_silver_clean", f"Numeric null handling complete — {dropped} rows dropped")


## 8. Enforce correct data types

In [0]:
#   postal_code → IntegerType - must be StringType (leading zeros, not a number)

df = df.withColumn(
    "postal_code",
    F.col("postal_code").cast("string")
)

# Verify the cast worked
postal_type = df.schema["postal_code"].dataType.simpleString()
print(f"  postal_code type after cast : {postal_type}")

if postal_type == "string":
    print("  ✅ postal_code correctly cast to StringType")
else:
    log("WARN", "02_silver_clean", f"postal_code cast may have failed: {postal_type}")


## 9. Standardise categorical string columns

In [0]:
# Problems this fixes:
#   "  West  " (leading/trailing spaces) → "West"
#   "WEST" or "west"                     → "West"  (Title Case for categories)
#   "consumer" vs "Consumer"             → "Consumer"
#
# We apply:
#   trim()       → remove leading/trailing whitespace
#   initcap()    → Title Case for human-readable categories
#
# Columns that get standardised:
#   segment, region, category, sub_category, ship_mode, city, state, country
#
# Columns we do NOT initcap:
#   customer_name  → already proper names, initcap is fine
#   product_name   → may have acronyms (e.g. "HP" → "Hp"), leave as-is
#   order_id       → identifier, leave exact
#   customer_id    → identifier, leave exact
#   product_id     → identifier, leave exact

categorical_cols = [
    "segment", "region", "category",
    "sub_category", "ship_mode", "city",
    "state", "country"
]

for col_name in categorical_cols:
    df = df.withColumn(
        col_name,
        F.initcap(F.trim(F.col(col_name)))
    )

# Trim identifiers without changing case
id_cols = ["order_id", "customer_id", "product_id",
           "customer_name", "product_name"]

for col_name in id_cols:
    df = df.withColumn(col_name, F.trim(F.col(col_name)))

print("  ✅ String standardisation complete")
print(f"  initcap + trim applied to : {categorical_cols}")
print(f"  trim only applied to      : {id_cols}")


## 10. Business rule validation

In [0]:
# These are rules that must be true for the data to make business sense.
# Violations are flagged and investigated — not silently dropped.
#
# Rules:
#   1. sales must be > 0         (a transaction must have positive revenue)
#   2. quantity must be >= 1     (you can't sell zero or negative units)
#   3. discount between 0 and 1  (discount is a fraction: 0.2 = 20% off)
#   4. ship_date >= order_date   (you can't ship before the order is placed)

print("=== BUSINESS RULE VALIDATION ===\n")

rules = {
    "sales > 0"              : df.filter(F.col("sales") <= 0).count(),
    "quantity >= 1"          : df.filter(F.col("quantity") < 1).count(),
    "discount between 0-1"   : df.filter(
                                    (F.col("discount") < 0) |
                                    (F.col("discount") > 1)
                               ).count(),
    "ship_date >= order_date": df.filter(
                                    F.col("ship_date") < F.col("order_date")
                               ).count(),
}

all_passed = True
for rule, violation_count in rules.items():
    status = "✅" if violation_count == 0 else f"⚠️  {violation_count} violations"
    print(f"  {rule:<35} : {status}")
    if violation_count > 0:
        all_passed = False
        log("WARN", "02_silver_clean", f"Rule violation: {rule} → {violation_count} rows")

if all_passed:
    print("\n  ✅ All business rules passed")
else:
    print("\n  ⚠️  Review violations above before proceeding to Gold")


## 11. Add Silver-layer audit columns

In [0]:
# Silver metadata is richer than Bronze — we record:
#   _silver_processed_at  : when Silver cleaning ran
#   _silver_version       : version of the cleaning logic (for reprocessing)
#   _row_quality          : a quality flag for downstream consumers

df_silver = df.withColumns({
    "_silver_processed_at" : F.current_timestamp(),
    "_silver_version"      : F.lit("1.0"),
    "_row_quality"         : F.lit("clean"),
})

print(f"  Total columns in Silver : {len(df_silver.columns)}")
print(f"  ✅ Silver metadata added")


## 12. Write cleaned data to Silver Delta Table

In [0]:
# mode("overwrite") — safe because Silver is always fully reprocessed from Bronze.
# In a streaming or incremental pipeline you would use MERGE instead.

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

log("INFO", "02_silver_clean", f"Silver Delta Table written: {SILVER_TABLE}")


## 13. Comparison of before/after cleanup

In [0]:
# Cell 14: The before/after story — did cleaning work?
#
# A professional cleaning notebook always ends with a comparison that
# proves the cleaning had the intended effect.

df_silver_verify = spark.table(SILVER_TABLE)

bronze_rows = df_bronze.count()
silver_rows = df_silver_verify.count()
dropped     = bronze_rows - silver_rows

print("=" * 55)
print("  BRONZE → SILVER COMPARISON")
print("=" * 55)
print(f"  Bronze rows (raw)         : {bronze_rows:,}")
print(f"  Silver rows (clean)       : {silver_rows:,}")
print(f"  Rows removed              : {dropped:,}")
print(f"  Retention rate            : {silver_rows/bronze_rows*100:.2f}%")
print("=" * 55)

# Null check after cleaning — should be dramatically improved
string_cols_silver = {
    f.name for f in df_silver_verify.schema.fields
    if isinstance(f.dataType, StringType)
    and not f.name.startswith("_")
}

null_exprs_after = [
    F.count(
        F.when(
            F.col(c).isNull() | (F.col(c) == "") if c in string_cols_silver
            else F.col(c).isNull(),
            c
        )
    ).alias(c)
    for c in df_silver_verify.columns
    if not c.startswith("_")
]

null_counts_after = df_silver_verify.select(null_exprs_after).collect()[0]
remaining_nulls   = sum(null_counts_after)

print(f"  Remaining null/blank cells: {remaining_nulls}")
if remaining_nulls == 0:
    print("  ✅ All nulls resolved in Silver")
else:
    print("  ⚠️  Some nulls remain — review cleaning logic")
print("=" * 55)

log("INFO", "02_silver_clean", "Silver cleaning verification complete")


## 14. Notebook completion summary

In [0]:
# Cell 15: Notebook completion summary

print("=" * 55)
print("  SILVER CLEANING — COMPLETE")
print("=" * 55)
print(f"  Input  : {BRONZE_TABLE}")
print(f"  Output : {SILVER_TABLE}")
print(f"  Rows   : {bronze_rows:,} → {silver_rows:,}")
print(f"  Steps  : dedup → null handling → type casting")
print(f"           → standardisation → rule validation")
print("=" * 55)

log("INFO", "02_silver_clean", "Notebook 02 complete ✅")